# 01 Explore the data

Goal of this notebook: before building any model, find out what the data looks like.
- How many wafers, how big is one map, what do the numbers mean?
- What do the defect patterns look like?
- How many examples of each class are there?

Dataset: MixedWM38, see `data/README.md` for where to download it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from collections import Counter

print("PyTorch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 1. Load the file

An `.npz` file is a zip with numpy arrays in it. This one has two:
`arr_0` holds the wafer maps, `arr_1` the labels.

In [ ]:
# Relative path: the notebook runs from the notebooks folder, the data sits one folder up.
data = np.load("../data/Wafer_Map_Datasets.npz")
print(data.files)

maps = data["arr_0"]    # the wafer maps
labels = data["arr_1"]  # 8 labels per wafer

print("maps:  ", maps.shape, maps.dtype)
print("labels:", labels.shape, labels.dtype)

## 2. What do the numbers in a map mean?

Expected: 0 = no die (outside the round wafer), 1 = good die, 2 = failed die.
Check it instead of believing it.

In [ ]:
values, counts = np.unique(maps, return_counts=True)
for v, c in zip(values, counts):
    print(f"value {v}: {c:,} pixels")


## 3. Turn the label into a name

Each label is 8 numbers, one per single defect, 1 if it is on the wafer.
All zeros means a normal wafer. More than one 1 means a mixed defect.

In [ ]:
# The column order comes from the dataset description.
# DEFECTS is defined here, but the actual label values come from the dataset.
DEFECTS = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-Full", "Scratch", "Random"]

def label_name(label):
    # Collect the names of every defect that is switched on.
    present = [DEFECTS[i] for i in range(len(DEFECTS)) if label[i] == 1]
    if not present:
        return "Normal"
    return " + ".join(present)

print(labels[1], "->", label_name(labels[1]))
#for l in labels: print(l, "->", label_name(l))

## 4. Look at one wafer

In [ ]:
# Three colours for the three values: background, good, failed.
def show_wafer(ax, wafer_map, title):
    #matplotlib.axes.Axes.imshow -> Display data as an image, i.e., on a 2D regular raster.
    ax.imshow(wafer_map, cmap="viridis", vmin=0, vmax=2) #2d data, default image mapper
    ax.set_title(title, fontsize=12)
    ax.axis("off")

# fig is for when you want to save the image, control overall layout, or add multiple subplots
fig, ax = plt.subplots(figsize=(3, 3))
show_wafer(ax, maps[0], label_name(labels[0]))
plt.show()

## 5. One example of every single defect

Pick the first wafer that has exactly that one defect, so you see each pattern on its own.

In [ ]:
fig, axes = plt.subplots(1, len(DEFECTS) + 1, figsize=(16, 2.5))

# Normal first: a wafer with no defect at all.
normal = np.where(labels.sum(axis=1) == 0)[0][0]
show_wafer(axes[0], maps[normal], "Normal")

for i, name in enumerate(DEFECTS):
    # Exactly one defect, and it is this one.
    only_this = np.where((labels.sum(axis=1) == 1) & (labels[:, i] == 1))[0]
    show_wafer(axes[i + 1], maps[only_this[0]], name)

plt.show()

## 6. How many of each class?

Every combination of defects counts as its own class. The paper says 38.
If some classes are much smaller than others, the model will have trouble with them later.

In [ ]:
names = [label_name(l) for l in labels]
class_counts = Counter(names)

print("number of classes:", len(class_counts))
for name, n in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"{n:6,}  {name}")

In [ ]:
# The same as a chart, sorted from big to small.
sorted_counts = sorted(class_counts.items(), key=lambda x: -x[1])
plt.figure(figsize=(12, 8))
plt.bar([n for n, _ in sorted_counts], [c for _, c in sorted_counts])
plt.xticks(rotation=90, fontsize=12)
plt.ylabel("wafers")
plt.title("Wafers per class")
plt.tight_layout()
plt.show()

## 7. How many defects per wafer?

In [ ]:
per_wafer = Counter(labels.sum(axis=1).astype(int))
for k in sorted(per_wafer):
    print(f"{k} defect(s): {per_wafer[k]:,} wafers")

## What I noticed

Write down what you saw, in your own words. This is input for your analyse and advise document.

-